In [1]:
%load_ext autoreload
%autoreload 2

# Tabel 010 kvantorid 1

Lisaandmetena kasutatakse skripriga 910 kokku kogutud lemma pos korpuses esinemise statistikat.

Õpikulausete korpusest kogutakse kokku tipud - ülemus + vahetu alluv, kus:
* ülemuse sõnaliik on `NOUN` ja kääne `part` (p);
* alluv eelneb lauses ülemusele;
* alluva sünrel on `nmod`, sõnaliik on `NOUN` ja kääne `nom` või `gen` või `part`

**Ülesande originaalpüstitus**

Otsime õpikulausete korpusest ülemuse-alluva paare (nt tass kohvi, tassist kohvist, tassi kohviga), kus
1. ülemuse sõnaliik = NOUN ja kääne = p (part) + alluva sünrel = nmod ja kääne = n (nom), p (part) või g (gen)

Tulemuste tabelis võiksid olla järgmised veerud: alluva lemma, alluva kääne, alluva arv, ülemuse lemma, ülemuse kääne, ülemuse arv, kogu lause, ?päringule vastav fragment, alluva lemma koguarv korpuses.

**Tulemus**

Tulemuseks on tabel tsv formaadis.


Tabeli veerud
||||
|---|---|---|
|**child_lemma**| alluva lemma |---|
|**child_case**| alluva kääne |---|
|**child_number**| alluva arv |---|
|**parent_lemma**| ülemuse lemma |---|
|**parent_case**| ülemuse käänel |---|
|**parent_number**| ülemuse arv |---|
|**text**| ?päringule vastav fragment |---|
|**sentence**| teve lause tekst, kus ülemus ja alluv toodetud esile alakriipsudega  \_\_sõne\_\_ |---|
|**sentence_id**| lause id koondkorpuse andmebaasis|---|
|**child_lemma_total**| lemma + POS esinemise arv Õpikulausete korpuses |---|


In [2]:
from notebook_context import LISTS_FOLDER, corpus_reader

import pandas as pd
from datetime import datetime

FIELDNAMES = [
    "child_lemma",
    "child_pos",
    "child_case",
    "child_number",
    "parent_lemma",
    "parent_pos",
    "parent_case",
    "parent_number",
    "text",
    "sentence",
    "sentence_id",
    "child_lemma_total",
]

LEMMAS_STAT = LISTS_FOLDER / "stats/lemmas.tsv"
TYPE = "quantifier_1"
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

OUT_FILE = LISTS_FOLDER / "results" / f"{TYPE}_{date_time}.tsv"

In [3]:
# loeme sisse lemmade statistika ja teeme vastava dict
df_lemmas = pd.read_csv(LEMMAS_STAT, sep="\t")
lemmas_stat = {f"{row['lemma']}\t{row['POS']}": int(row['total']) for _, row in df_lemmas.iterrows()}
lemmas_stat['olema\tVERB']

629

In [4]:
%%time

collocations = []
count = 0
for sentence_id, graph in corpus_reader.get_sentences():
    count += 1
    if not sentence_id:
        sentence_id = count

    # matrix for node distances
    dpath = graph.get_distances_matrix()

    # noun nodes
    noun_nodes = graph.get_nodes_by_attributes(attrname="POS", attrvalue="NOUN")

    # nmod
    nmod_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="nmod")

    # iteratsioon üle nimisõnade
    for noun in noun_nodes:
        noun_lemma = graph.nodes[noun]["lemma"]
        noun_pos = graph.nodes[noun]["POS"]
        noun_case = graph.get_node_case(noun)
        noun_number = graph.get_node_number(noun)

        if noun_case not in ["Par"]:
            continue
        # childnodes
        kids = [k for k in dpath[noun] if dpath[noun][k] == 1]

        # iterate over nmod children
        for nmod in nmod_nodes:
            # kui pole vahetu alluv, siis ei huvita
            if nmod not in kids:
                continue

            # nmod peab olema lauses enne ülemust
            if nmod > noun:
                continue
            nmod_case = graph.get_node_case(nmod)
            nmod_pos = graph.nodes[nmod]["POS"]
            if nmod_case not in ["Nom", "Gen", "Par"]:
                continue
            if nmod_pos != "NOUN":
                continue

            nmod_lemma = graph.nodes[nmod]["lemma"]
            nmod_number = graph.get_node_number(nmod)

            words = []
            for n in sorted(graph.nodes):
                if not n:
                    continue
                if n in (noun, nmod):
                    words.append(f'___{graph.nodes[n]["form"]}___')
                else:
                    words.append(graph.nodes[n]["form"])
            sentence_text = " ".join(words)

            text = " ".join([graph.nodes[n]["form"] for n in sorted((noun, nmod))])

            collocations.append(
                (
                    nmod_lemma,  # child_lemma
                    nmod_pos,  # child_pos
                    nmod_case,  # child_case
                    nmod_number,  # child_number
                    noun_lemma,  # parent_lemma
                    noun_pos,  # parent_pos
                    noun_case,  # parent_case
                    noun_number,  # parent_number
                    text,  # text
                    sentence_text,  # sentence
                    sentence_id,  # sentence_id
                    lemmas_stat.get(f"{nmod_lemma}\t{nmod_pos}", 0),  # child_lemma_total
                )
            )

print(f"sentences processed: {count}")
print(f"collocations found: {len(collocations)}")

../data/Model2Eesti-keele-kui-teise-keele-kooliõpikute-lausete-korpus-2021.conllu
sentences processed: 42279
collocations found: 1034
CPU times: user 4.42 s, sys: 52 ms, total: 4.47 s
Wall time: 4.64 s


In [6]:


df = pd.DataFrame(collocations, columns=FIELDNAMES)
df.to_csv(OUT_FILE, sep="\t", index=None)
df.head()

,child_lemma,child_pos,child_case,child_number,parent_lemma,parent_pos,parent_case,parent_number,text,sentence,sentence_id,child_lemma_total
0,gümnaasium,NOUN,Gen,Sing,lõpetamine,NOUN,Par,Sing,gümnaasiumi lõpetamist,"Üha rohkem leidub noori , kes pärast ___gümnaa...",9,37
1,aine_tund,NOUN,Gen,Plur,kokku_langevus,NOUN,Par,Plur,ainetundide kokkulangevusi,Tegelikult saab vaba ainevalikut rakendada vai...,65,1
2,kannatlikkus,NOUN,Par,Sing,arvestamine,NOUN,Par,Sing,kannatlikkust arvestamist,Olen õppinud ___kannatlikkust___ ja teistega _...,71,1
3,nädal,NOUN,Nom,Sing,aeg,NOUN,Par,Sing,nädal aega,Kolmandas klassis ( meie 9. kl ) viibitakse __...,136,225
4,põhi_kool,NOUN,Gen,Sing,lõpetamine,NOUN,Par,Sing,põhikooli lõpetamist,"Tänu sellele oli meie klassis üks õpilane , ke...",139,23
